## Sensitivity Analysis of W&B Runs

In [3]:
import wandb

# Connect to Wandb
wandb.login()

# Specify the project name
project_name = "karman"

# Specify the date after which you want to retrieve the run IDs
start_date = "2024-08-04T19:12:00"      # August 4th, 2024 at 8:12 pm

# Get the runs for the specified project and date
api = wandb.Api()

runs = api.runs(path="karman", filters={"createdAt": {"$gte": start_date}}) #, created_after=start_date)

# Extract the run IDs
run_ids = [run.id for run in runs]

# Print the run IDs
print(run_ids)

# Check these are all past start_date, will error if not
assert(all([r.createdAt > start_date for r in runs]))

['smz6tk2i', 'ws3cfslm', '6017f83e', 'apz3jl5g', '4q6otk4j', 'fpbxh2w8', '9cpg011x', 'roevlgyc', '376nuxmz', '7uwm8hmq', 't1irwkmt', 'vivdyfg3', 'eczj9h30', 'pj2nxddx', 'ksokuul0', '1epcmrr7', 'p9s2zqsp', 'yxayzyjv', '1vyweybe', 'v5cs6szr', 'm2p6h656', '0mdzrxlk', '8g0pysdr', '5ohv0f15', 'iok3557p', '1ct7n7yy', 'uthb5v1v', 'p4xe3b2r', 'ts9xs3wo', 'zb61zyhg', 'zqqm7ned', 'r95l506l', 'xcour2n0', 'ba0elg1q', '8sv64aqn', 'i45e8sw0', 'x76t0p07', 'a0lz32if', 'szgfk166', 'm0zvxq1e', 'wdlf5xfp', 'r6ht7q9h', 'sxp477yk', '08w0skva', '91ih08zk']


In [4]:
run = runs[0]
valid_keys = [k for k in run.summary.keys() if "valid_total" in k]
valid_history = run.history(keys=["_step"]+valid_keys) 

train_keys = [k for k in run.summary.keys() if "train_total" in k]
train_history = run.history(keys=["_step"]+train_keys) 

,_step,nn_mape_valid_total,nn_mse_valid_total,nrlmsise00_mse_valid_total,nrlmsise00_mape_valid_total
0,28202,18.723280,0.002387,0.008957,48.824759
1,56406,16.814975,0.002254,0.008957,48.823447


In [42]:
for c in train_history.columns:
    if c == "_step":
        continue
    print(c, train_history[c].min(), train_history[c].argmin())
    # print(train_history[c][], train_history[c].argmax())

_step 25654 0
q_loss_train_total 0.19893452633816147 0
nn_mape_train_total 18.062211990356445 8
nrlmsise00_mape_train_total 47.16281509399414 2
nn_mse_train_total 0.0024647616039170494 8
nrlmsise00_mse_train_total 0.00835809763520956 3


In [6]:
import pandas as pd

run = runs[0]
valid_keys = [k for k in run.summary.keys() if "valid_total" in k]
train_keys = [k for k in run.summary.keys() if "train_total" in k]

# Create an empty DataFrame to store the results
results_df = pd.DataFrame(columns=['Run ID'] + train_keys + valid_keys)

results = []

# Iterate over each run
for run in runs:
    # Get the train_history and valid_history for the run
    train_history = run.history(keys=["_step"]+train_keys)
    valid_history = run.history(keys=["_step"]+valid_keys)

    # Should really do this with argmin on one and take the values from the rest!!
    train_arg_min = train_history["nn_mape_train_total"].argmin()
    valid_arg_min = valid_history["nn_mape_valid_total"].argmin()

    result = {'Run ID': run.id, "Run Name": run.name.replace("run_gpu", "").replace("_epochs_"+str(run.config["epochs"]), ""), "Lag": None, "Epochs": run.config["epochs"],
                                    **{key: train_history[key][train_arg_min] for key in train_keys if key not in ["_step", "q_loss_train_total", "q_loss_valid_total"]},
                                    **{key: valid_history[key][valid_arg_min] for key in valid_keys if key not in ["_step", "q_loss_train_total", "q_loss_valid_total"]}
    }
    if "lag_minutes" in run.config:
        result["Lag"] = run.config["lag_minutes"]

    results.append(result)

# Print the results
# print(results_df)
pd.DataFrame(results).sort_values("nn_mape_valid_total")

Empty DataFrame
Columns: [Run ID, nn_mape_train_total, nrlmsise00_mse_train_total, nn_mse_train_total, nrlmsise00_mape_train_total, nn_mape_valid_total, nn_mse_valid_total, nrlmsise00_mse_valid_total, nrlmsise00_mape_valid_total]
Index: []


,Run ID,Run Name,Lag,Epochs,nn_mape_train_total,nrlmsise00_mse_train_total,nn_mse_train_total,nrlmsise00_mape_train_total,nn_mape_valid_total,nn_mse_valid_total,nrlmsise00_mse_valid_total,nrlmsise00_mape_valid_total
38,szgfk166,run_tft_first_w_soho_and_w_omni_and_proxies,500.0,20,11.760783,0.008358,0.001111,47.163149,13.100392,0.001429,0.008957,48.822879
28,ts9xs3wo,run_tft_everything,500.0,20,11.861155,0.008358,0.001133,47.163406,13.719629,0.001545,0.008957,48.825405
25,1ct7n7yy,_tft_w_omni_and_soho_wo_indices_and_proxies_w_...,60000.0,20,11.569569,0.008358,0.001101,47.163433,13.763132,0.001516,0.008957,48.824276
40,wdlf5xfp,_tft_wo_sw_all_except_s107_30_epochs,500.0,30,11.997643,0.008358,0.001142,47.163498,14.292094,0.001614,0.008957,48.824821
14,ksokuul0,_tft_w_omni_and_soho_wo_indices_and_proxies_w_...,50000.0,20,12.373112,0.008358,0.001248,47.162933,14.366497,0.001656,0.008957,48.824631
19,v5cs6szr,_tft_w_omni_and_soho_wo_indices_and_proxies_w_...,90000.0,20,12.592281,0.008358,0.001289,47.163254,14.376472,0.001633,0.008957,48.822826
37,a0lz32if,run_tft_wo_omni_except_omni_indices_and_wo_ap_...,500.0,20,12.639576,0.008358,0.001264,47.163284,14.408853,0.001637,0.008957,48.825375
20,m2p6h656,_tft_w_omni_and_soho_wo_indices_and_proxies_w_...,80000.0,20,12.725663,0.008358,0.001310,47.163128,14.487440,0.001669,0.008958,48.827282
17,yxayzyjv,_tft_w_omni_and_soho_wo_indices_and_proxies_w_...,70000.0,20,13.012463,0.008358,0.001367,47.163269,14.695791,0.001641,0.008958,48.826965
21,0mdzrxlk,_tft_w_omni_and_soho_wo_indices_and_proxies_w_...,20000.0,20,12.135327,0.008358,0.001201,47.162727,14.808370,0.001729,0.008956,48.821312


In [7]:
pd.DataFrame(results).sort_values("nn_mape_valid_total").to_csv("results.csv")

In [20]:
run.config

{'lr': 0.001,
 'device': 'cuda:0',
 'epochs': 30,
 'dropout': 0.05,
 'max_date': '2024-05-31 23:59:32',
 'min_date': '2000-07-29 00:59:47',
 'run_name': 'run_gpu_tft_soho_only_w_40000_lag_100_resolution',
 'goes_path': None,
 'soho_path': '../data/soho_data/soho_data.csv',
 'batch_size': 64,
 'model_path': None,
 'state_size': 64,
 'torch_type': 'float32',
 'lag_minutes': 40000,
 'lstm_layers': 2,
 'num_workers': 16,
 'thermo_path': '../data/satellites_data_w_sw_2mln.csv',
 'wandb_inactive': False,
 'attention_heads': 4,
 'nrlmsise00_path': '../data/nrlmsise00_data/nrlmsise00_time_series.csv',
 'omni_indices_path': 'None',
 'resolution_minutes': 100,
 'omni_solar_wind_path': 'None',
 'normalization_dict_path': None,
 'omni_magnetic_field_path': 'None',
 'features_to_exclude_thermo': 'celestrack__ap_average__,JB08__d_st_dt__[K],space_environment_technologies__f107_obs__,space_environment_technologies__f107_average__,space_environment_technologies__s107_obs__,space_environment_technologi

In [19]:
runs[0].config['lag_minutes']

500

In [ ]:
import pandas as pd

# Create an empty DataFrame to store the results
results_df = pd.DataFrame(columns=['Run ID'] + train_keys + valid_keys)

# Iterate over each run
for run in runs:
    # Get the train_history and valid_history for the run
    train_history = run.history(keys=["_step"]+train_keys)
    valid_history = run.history(keys=["_step"]+valid_keys)

    results_df = results_df.append({'Run ID': run.id, 
                                    **{key: train_history[key].min() for key in train_keys if key not "_step"},
                                    **{key: valid_history[key].min() for key in valid_keys if ket not "_step"}
    }, ignore_index=True)

# Print the results
print(results_df)

In [29]:
history = run.scan_history(keys=["_step", "nn_mape_valid"])
# for i in history:
#     if i["_step"] == 100:
#         print(i["nn_mape_train"])#, i["nn_mape_valid"])
#         break
history.shape

AttributeError: 'SampledHistoryScan' object has no attribute 'shape'

In [28]:
history.shape

AttributeError: 'SampledHistoryScan' object has no attribute 'shape'

In [12]:
[k for k in runs[0].summary.keys() if "_total" in k]

['nn_mape_valid_total',
 'nrlmsise00_mse_valid_total',
 'q_loss_valid_total',
 'q_loss_train_total',
 'nn_mape_train_total',
 'nrlmsise00_mape_train_total',
 'nrlmsise00_mape_valid_total',
 'nn_mse_train_total',
 'nn_mse_valid_total',
 'nrlmsise00_mse_train_total']

In [11]:
run.summary["nn_mape_valid_total"]


23.732192993164062